# Notebook 05 · Evaluación final y análisis de errores

**Autores:** Isabel Cristina Morales Parra – Jaime Esteban Bejarano Caldas

## 0. Validación temporal (extensión experimental)

En el notebook 01 se identificó una deriva temporal: la proporción de reseñas negativas cae de 7,7 % (2017) a 1,5 % (2024). Una partición aleatoria mezcla años, de modo que el modelo se entrena y evalúa con reseñas de todas las épocas. En un uso real, en cambio, el modelo se entrena con reseñas **pasadas** y clasifica reseñas **futuras**.

Se modifica una sola condición respecto del notebook 02, el criterio de partición:
- **Partición aleatoria estratificada** (notebook 02): 80 % / 20 %.
- **Partición temporal:** entrenamiento con 2014–2021 y prueba con 2022–2024.

Se usa el mejor modelo (TF-IDF con bigramas, regresión logística, `C = 30`).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.append(str(RAIZ))
from src.evaluacion import metricas_clasificacion

RUTA_FIGURAS, RUTA_RESULTADOS, RUTA_MODELOS = RAIZ / "figures", RAIZ / "results", RAIZ / "models"
RUTA_MODELOS.mkdir(exist_ok=True)
SEMILLA = 42
sns.set_theme(style="whitegrid", palette="deep")

df_en = pd.read_csv(RAIZ / "data" / "processed" / "resenas_en_depuradas.csv")
corpus_A = df_en.dropna(subset=["polaridad_A"]).reset_index(drop=True)

def modelo_tfidf():
    return Pipeline([
        ("representacion", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, token_pattern=r"\S+")),
        ("clasificador", LogisticRegression(class_weight="balanced", solver="liblinear", C=30, max_iter=2000)),
    ])

# --- Partición aleatoria (la del notebook 02) ---
X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(
    corpus_A["texto_limpio"], corpus_A["polaridad_A"], test_size=0.20,
    stratify=corpus_A["polaridad_A"], random_state=SEMILLA)
modelo_aleatorio = modelo_tfidf().fit(X_entrena, y_entrena)

# --- Partición temporal ---
pasado = corpus_A[corpus_A["anio"] <= 2021]
futuro = corpus_A[corpus_A["anio"] >= 2022]
modelo_temporal = modelo_tfidf().fit(pasado["texto_limpio"], pasado["polaridad_A"])

validacion = pd.DataFrame({
    "Aleatoria (80/20)": metricas_clasificacion(y_prueba, modelo_aleatorio.predict(X_prueba)),
    "Temporal (2014–2021 → 2022–2024)": metricas_clasificacion(futuro["polaridad_A"], modelo_temporal.predict(futuro["texto_limpio"])),
}).T
validacion["Reseñas de entrenamiento"] = [len(X_entrena), len(pasado)]
validacion["Reseñas de prueba"] = [len(X_prueba), len(futuro)]
validacion["Negativas en prueba"] = [(y_prueba == "negativo").sum(), (futuro["polaridad_A"] == "negativo").sum()]
display(validacion.round(3))
validacion.to_csv(RUTA_RESULTADOS / "05_validacion_temporal.csv")

# --- Se guarda el modelo final para la función predict_sentiment ---
joblib.dump(modelo_aleatorio, RUTA_MODELOS / "tfidf_logistica.joblib")

,F1-macro,Exactitud balanceada,Precisión (negativo),Recall (negativo),Exactitud (referencia),Reseñas de entrenamiento,Reseñas de prueba,Negativas en prueba
Aleatoria (80/20),0.918,0.927,0.824,0.862,0.987,13150,3288,130
Temporal (2014–2021 → 2022–2024),0.921,0.932,0.825,0.868,0.994,8705,7733,152


['c:\\Users\\isabe\\Cursos\\ip_2026_b\\NLP\\pln-luxuryhotels-sentimiento\\models\\tfidf_logistica.joblib']

**Resultado.** El modelo entrenado con reseñas de 2014–2021 clasifica las reseñas de 2022–2024 con un **F1-macro de 0,921**, prácticamente igual al de la partición aleatoria (0,918), con *recall* negativo de 0,868 frente a 0,862. **La deriva temporal no degrada el desempeño:** el vocabulario con que se expresa la insatisfacción (*not clean*, *rude*, *refund*) es estable en el tiempo, aunque la proporción de reseñas negativas disminuya.

Esto ocurre a pesar de que el modelo temporal se entrenó con un 34 % menos de reseñas (8.705 frente a 13.150). Según la curva de aprendizaje del notebook 03, con más de 8.000 reseñas TF-IDF ya está cerca de su techo.

**Matices.**
- La exactitud sube a 0,994 en la partición temporal solo porque el periodo de prueba tiene menos negativas (2,0 % frente a 4,0 %). Es un nuevo ejemplo de por qué la exactitud no sirve como métrica principal.
- La prueba temporal cuenta con 152 reseñas negativas, de modo que la comparación tiene la misma incertidumbre que la del notebook 02.
- La estabilidad observada aplica al periodo estudiado. Un cambio de contexto (nuevos hoteles, otro destino, otra plataforma) podría modificar el vocabulario y requeriría reentrenar el modelo.

## 1. Análisis de errores

Se examinan los errores del mejor modelo (TF-IDF, partición aleatoria) en el conjunto de prueba:
- **Falsos negativos:** reseñas negativas clasificadas como positivas.
- **Falsos positivos:** reseñas positivas clasificadas como negativas.

Para cada tipo se muestran los casos en los que el modelo se equivocó **con más confianza**, y se compara la longitud de las reseñas bien y mal clasificadas. Al final se presenta la función reutilizable `predict_sentiment`.

In [2]:
pd.set_option("display.max_colwidth", 400)

prueba = corpus_A.loc[X_prueba.index].copy()
prueba["prediccion"] = modelo_aleatorio.predict(X_prueba)
prueba["p_negativo"] = modelo_aleatorio.predict_proba(X_prueba)[:, 0]   # columna 0 = 'negativo'
prueba["error"] = prueba["polaridad_A"] != prueba["prediccion"]

falsos_neg = prueba[(prueba["polaridad_A"] == "negativo") & (prueba["prediccion"] == "positivo")].sort_values("p_negativo")
falsos_pos = prueba[(prueba["polaridad_A"] == "positivo") & (prueba["prediccion"] == "negativo")].sort_values("p_negativo", ascending=False)

print(f"Falsos negativos: {len(falsos_neg)}  |  calificaciones: {falsos_neg['Rating'].value_counts().to_dict()}")
print(f"Falsos positivos: {len(falsos_pos)}  |  calificaciones: {falsos_pos['Rating'].value_counts().to_dict()}")
print("\nMediana de palabras — bien clasificadas vs. errores:")
display(prueba.groupby("error")["n_palabras"].median().rename({False: "Bien clasificadas", True: "Errores"}).to_frame())

print("\nFALSOS NEGATIVOS más confiados (negativas que el modelo vio como positivas):")
display(falsos_neg[["Rating", "p_negativo", "Review"]].head(4))
print("\nFALSOS POSITIVOS más confiados (positivas que el modelo vio como negativas):")
display(falsos_pos[["Rating", "p_negativo", "Review"]].head(4))

pd.concat([falsos_neg.assign(tipo="falso negativo"), falsos_pos.assign(tipo="falso positivo")])[
    ["tipo", "Rating", "p_negativo", "Review"]].to_csv(RUTA_RESULTADOS / "05_errores_tfidf.csv", index=False)

Falsos negativos: 18  |  calificaciones: {2: 12, 1: 6}
Falsos positivos: 24  |  calificaciones: {4: 18, 5: 6}

Mediana de palabras — bien clasificadas vs. errores:


,n_palabras
error,
Bien clasificadas,58.0
Errores,90.0



FALSOS NEGATIVOS más confiados (negativas que el modelo vio como positivas):


,Rating,p_negativo,Review
6694,2,0.007764,"I was very disappointed with our stay here, my experience of the front desk staff was surprising, they were rude to me while we resolving an issue around our stay. The room was clean, nice amenities but it’s a large hotel and many rooms font have a good view.The staff in both the restaurant and the upstairs lounge were amazing though! Warm, fun and friendly. The food was excellent and the serv..."
8275,2,0.012722,The wifi is not good at all especially in the 6th floor The food is average but good comparing to Kathmandu food quality. The staff were nice and helped us in all requests but they need more training in the customer service The location is good
8171,2,0.026231,"Overall, the hotel is quite comfortable, with a full range of amenities, and in a good location. However, their security was extremely slow to respond to a security incident at the hotel, leaving my colleagues and me feeling unsafe."
1427,2,0.029799,The staff & food were the best but the hotel itself is very old and unclean. Felth smell is very where. The internet is disconnected regularly.Nothing to do in surrounding area.Has 3 restaurants. I liked Indian and Chinese.



FALSOS POSITIVOS más confiados (positivas que el modelo vio como negativas):


,Rating,p_negativo,Review
13809,4,0.996360,Mistic mountain Plus points Very beautifulAt top of hillAmazing view of HimalayasGood size roomsBeautiful roomsThey have Pool and JacuzziBut these were not operational during our stay .no one informed us beforehand .Minus Poor availability of hot waterVery limited veg food options.v far from market so one is forced to eat hereToo overpriced for everything plus huge taxes Staff not v friendly ...
10323,4,0.922274,"What I liked:- Good location, access to shops, restaurants outside the hotel- Attached casino- Overral service- great breakfast that was included with many choices- internet speed was decent, not fastIssues:- room is equivalent to a 3-star in Europe/US comparison but great overral- the pictures you see online for grand staircase and hallways are of the casino not hotel, hotel is a bit simpler-..."
1599,4,0.913247,"Breakfast and staff in breakfast areas are not up to 5 star standard. Manager and staff there are not welcoming, their rare smiles looked cold and forced. Food items were not fresh, not replenished regularly and not kept hot. Croissants were stale and hard. Breakfast place is far from the rooms and problematic to reach when it rains. The shower head does not work well. TV worked half of the t..."
13957,4,0.872608,We visited there with our big group 32 pax with 13 rooms reservation on 21 Sep 2019. Hotel property and view form the hotel is awesome . Staffs are nice and helpful . But I had very bad experience of slow check in service . We reached there by 3 Pm and we were checked in at the same time . But they gave us room almost 45 minutes late . I asked the check in guy about the problem . He said check...


**Resultado.** El modelo comete 42 errores en 3.288 reseñas de prueba: 18 falsos negativos y 24 falsos positivos.

**1. Los errores se concentran en las calificaciones intermedias.** Los falsos negativos son sobre todo reseñas de **2 estrellas** (12 de 18), no de 1. Los falsos positivos son sobre todo de **4 estrellas** (18 de 24), no de 5. El modelo falla en la frontera, donde el texto es ambivalente.

**2. Los errores son reseñas más largas** (mediana de 90 palabras, frente a 58 en las bien clasificadas). Una reseña larga suele combinar elogios y críticas, como se vio en las reseñas con más de un tópico (notebook 04).

**3. Falsos negativos: críticas puntuales rodeadas de elogios.** Las reseñas negativas que el modelo vio como positivas con más confianza mezclan una queja grave con muchos elogios:
- «*rude* front desk… *I will not return*», pero también «*amazing*… *warm, fun and friendly*… *food was excellent*»;
- «*security incident*… *feeling unsafe*», pero «*quite comfortable*, *full range of amenities*, *good location*».

El modelo suma las señales de cada palabra (bolsa de palabras): muchos elogios pesan más que una queja, aunque la queja sea la que determinó la calificación. **No capta qué aspecto pesa más para el huésped.**

**4. Falsos positivos: etiquetas inconsistentes con el texto.** Las reseñas positivas que el modelo vio como negativas con más confianza (probabilidad de 0,87 a 0,996) son, en su contenido, **reseñas de queja con 4 estrellas**: piscina y jacuzzi fuera de servicio, sin agua caliente, aire acondicionado averiado, cobros ocultos en el bufé, desayuno «*not up to 5 star standard*», 45 minutos de espera en el check-in. **En estos casos, el modelo lee el texto mejor de lo que la calificación lo resume.** Es una limitación de usar la calificación como etiqueta: el huésped puede ser generoso con las estrellas y severo en el texto.

**Implicaciones.**
- Parte del error medido es **ruido de etiquetado**, no error del modelo. El desempeño real podría ser algo mayor que el reportado.
- Para un hotel, los falsos positivos son **valiosos**: son quejas concretas escondidas tras una buena calificación, que un análisis basado solo en estrellas pasaría por alto.
- **Trabajo futuro:** modelos de sentimiento por aspecto, que asignen una polaridad a cada aspecto mencionado, y modelos contextuales (*transformers*), que capten la negación y el contraste («*but*», «*however*») mejor que una bolsa de palabras.

In [4]:
from src.prediccion import predict_sentiment

ejemplos = [
    "The staff was incredibly friendly and the breakfast buffet was delicious. Amazing view of the mountains!",
    "The room was not clean, the shower did not work and they charged us extra without explanation.",
    "Nice location but the room was a bit old. Staff were helpful.",
]
display(pd.DataFrame([{"Reseña": t, **predict_sentiment(t)} for t in ejemplos]))

,Reseña,clase,probabilidad_negativo,texto_normalizado
0,The staff was incredibly friendly and the breakfast buffet was delicious. Amazing view of the mountains!,positivo,0.001,staff incredibly friendly breakfast buffet delicious amazing view mountain
1,"The room was not clean, the shower did not work and they charged us extra without explanation.",negativo,0.993,room not clean shower not work charge extra without explanation
2,Nice location but the room was a bit old. Staff were helpful.,positivo,0.003,nice location room bit old staff helpful


**Resultado.** La función `predict_sentiment` clasifica reseñas nuevas aplicando la misma normalización con la que se entrenó el modelo:
- La reseña de elogio se clasifica como **positiva** (probabilidad de negativo: 0,001).
- La reseña de queja se clasifica como **negativa** (0,993). La normalización conserva las negaciones (*room **not** clean shower **not** work*), que son la señal más fuerte del modelo.
- La reseña mixta («*Nice location but the room was a bit old*») se clasifica como **positiva** (0,003). Es coherente con el patrón del notebook 02: *a bit* atenúa la crítica y es propio de reseñas favorables.

La función está en `src/prediccion.py` y usa el modelo guardado en `models/tfidf_logistica.joblib`, que se regenera al ejecutar este notebook.